In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Setting Up Environment/venv

In [2]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-py38_23.11.0-2-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/miniconda3

PREFIX=/opt/miniconda3
Unpacking payload ...
                                                                                
Installing base environment...





Preparing transaction: done
Executing transaction: done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/miniconda3


In [3]:
!/opt/miniconda3/bin/conda create -n bevfusion python=3.8 -y
!/opt/miniconda3/envs/bevfusion/bin/pip install torch==1.10.2+cu113 torchvision==0.11.3+cu113 torchaudio==0.10.2 -f https://download.pytorch.org/whl/torch_stable.html
!/opt/miniconda3/envs/bevfusion/bin/python -c "import torch; print(torch.__version__)"

Channels:
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/miniconda3/envs/bevfusion

  added / updated specs:
    - python=3.8


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-5.1          |           52_gnu           7 KB
    ca-certificates-2026.8.13  |       h06a4308_0         107 KB
    ld_impl_linux-64-2.44      |       h9e0c5a2_3         725 KB
    libffi-3.4.8               |       h06d3fd0_3         137 KB
    libgcc-15.2.0              |       h69a1729_8         803 KB
    libgcc-ng-15.2.0           |       h166f726_8          28 KB
    libstdcxx-15.2.0           |       h39759b7_8         3.7 MB
    libstdcxx-ng-15.2.0        |       hc03a8fd_8          28 KB
    libxcb-1.17.0              |       h9b100fa_0         430 KB
    libzlib-1.3.2              |       h47b2149_0          60 KB
    ncurses-6.6   

## CUDA toolkit + gcc (needed BEFORE compiling BEVFusion's custom ops later)
Moved up here, right after torch, so everything is ready by the time we
reach `setup.py develop` — no more building mmdetection3d first and
discovering the toolkit/gcc are missing partway through.

In [4]:
!/opt/miniconda3/bin/conda install -n bevfusion -y -c nvidia/label/cuda-11.3.1 cuda-toolkit

Channels:
 - nvidia/label/cuda-11.3.1
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/miniconda3/envs/bevfusion

  added / updated specs:
    - cuda-toolkit


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    cuda-command-line-tools-11.3.1|       h712c49d_0           2 KB  nvidia/label/cuda-11.3.1
    cuda-compiler-11.3.1       |       h712c49d_0           2 KB  nvidia/label/cuda-11.3.1
    cuda-cudart-11.3.109       |       hfb95d0c_0         1.1 MB  nvidia/label/cuda-11.3.1
    cuda-cuobjdump-11.3.122    |       hbf6ec6b_0         115 KB  nvidia/label/cuda-11.3.1
    cuda-cupti-11.3.111        |       h12ad217_0        19.5 MB  nvidia/label/cuda-11.3.1
    cuda-cuxxfilt-11.3.122     |       h4dc11a3_0          32 KB  nvidia/label/cuda-11.3.1
    cuda-gdb-11.3.109          |       h33b7820_0        39.0 MB  nvidia/label/cuda-11.3.

In [5]:
!/opt/miniconda3/bin/conda install -n bevfusion -y -c conda-forge gcc_linux-64=9 gxx_linux-64=9

Channels:
 - conda-forge
 - defaults
 - nvidia/label/cuda-11.3.1
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/miniconda3/envs/bevfusion

  added / updated specs:
    - gcc_linux-64=9
    - gxx_linux-64=9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    binutils_impl_linux-64-2.39|       he00db2b_1         4.7 MB  conda-forge
    binutils_linux-64-2.39     |      h5fc0e48_13          27 KB  conda-forge
    gcc_impl_linux-64-9.5.0    |      h99780fb_19        43.5 MB  conda-forge
    gcc_linux-64-9.5.0         |      h4258300_13          29 KB  conda-forge
    gxx_impl_linux-64-9.5.0    |      h99780fb_19        10.6 MB  conda-forge
    gxx_linux-64-9.5.0         |      h43f449f_13          28 KB  conda-forge
    kernel-headers_linux-64-5.14.0|       he073ed8_3         1.3 MB  conda-forge
    ld_impl_linux-64-2.39      |       hcc3a1bd_1  

In [6]:
# Confirm the exact compiler binary names conda-forge installed before wiring
# them into CC/CXX below.
!ls /opt/miniconda3/envs/bevfusion/bin/ | grep gcc

x86_64-conda_cos6-linux-gnu-gcc
x86_64-conda_cos6-linux-gnu-gcc-ar
x86_64-conda_cos6-linux-gnu-gcc-nm
x86_64-conda_cos6-linux-gnu-gcc-ranlib
x86_64-conda-linux-gnu-gcc
x86_64-conda-linux-gnu-gcc-ar
x86_64-conda-linux-gnu-gcc-nm
x86_64-conda-linux-gnu-gcc-ranlib


In [8]:
# Persist these for the rest of the session (must be re-run after any kernel restart).
import os
os.environ["CUDA_HOME"] = "/opt/miniconda3/envs/bevfusion"
os.environ["PATH"] = "/opt/miniconda3/envs/bevfusion/bin:" + os.environ["PATH"]
# TODO: update these two filenames to exactly match the `ls | grep gcc` output above
os.environ["CC"] = "/opt/miniconda3/envs/bevfusion/bin/x86_64-conda_cos6-linux-gnu-gcc"
os.environ["CXX"] = "/opt/miniconda3/envs/bevfusion/bin/x86_64-conda_cos6-linux-gnu-g++"

In [9]:
# Both should now report versions from the env (11.3 and 9.x), not Kaggle's
# system defaults (12.8 / newer gcc).
!nvcc --version
!$CC --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2021 NVIDIA Corporation
Built on Wed_Jun__9_20:32:38_PDT_2021
Cuda compilation tools, release 11.3, V11.3.122
Build cuda_11.3.r11.3/compiler.30059648_0
x86_64-conda_cos6-linux-gnu-gcc (conda-forge gcc 9.5.0-19) 9.5.0
Copyright (C) 2019 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [10]:
!/opt/miniconda3/envs/bevfusion/bin/pip install -U openmim

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.1/953.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 76.8 MB/s eta 0:00:00
  Created wheel for oss2: filename=oss2-2.17.0-py3-none-any.whl size=112372 sha256=b5954ad4d3bd3e33d3289f23b3120bc95cd0d0ebf52194e1ba611d17cd4d5d8a
  Stored in directory: /root/.cache/pip/wheels/84/f1/a7/a20c3f53c7dbd8a1e5aa17ab1759465203b4ee46d6b3994dd2
  Created wheel for crcmod: filename=crcmod-1.7-py3-none-any.whl size=18835 sha256=a50941d5b3e6c124fec32dbac415b132561584367eec4b0dbfc3fae02ac4ce30
  Stored in directory: /root/.cache/pip/whe

In [11]:
!/opt/miniconda3/envs/bevfusion/bin/mim install mmengine

Looking in links: https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 84.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.8 MB/s eta 0:00:00


In [12]:
!/opt/miniconda3/envs/bevfusion/bin/mim install "mmcv==2.1.0"
!/opt/miniconda3/envs/bevfusion/bin/mim install "mmdet>=3.0.0"

Looking in links: https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 MB 8.0 MB/s eta 0:00:0000:0100:01
Looking in links: https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html
Ignoring mmcv: markers 'extra == "mim"' don't match your environment
Ignoring mmengine: markers 'extra == "mim"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 27.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.2 MB/s eta 0:00:00


In [13]:
!/opt/miniconda3/envs/bevfusion/bin/python -c "import mmcv, mmdet; print('mmcv', mmcv.__version__); print('mmdet', mmdet.__version__)"

mmcv 2.1.0
mmdet 3.3.0


# Setting up mmdetection3d

In [14]:
!git clone https://github.com/open-mmlab/mmdetection3d.git -b dev-1.x

Cloning into 'mmdetection3d'...
remote: Enumerating objects: 21045, done.
remote: Total 21045 (delta 0), reused 0 (delta 0), pack-reused 21045 (from 1)
Receiving objects: 100% (21045/21045), 20.61 MiB | 22.79 MiB/s, done.
Resolving deltas: 100% (14770/14770), done.


In [15]:
%cd mmdetection3d

/kaggle/working/mmdetection3d


In [16]:
!/opt/miniconda3/envs/bevfusion/bin/pip install -v -e .

Using pip 24.2 from /opt/miniconda3/envs/bevfusion/lib/python3.8/site-packages/pip (python 3.8)
Obtaining file:///kaggle/working/mmdetection3d
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-lmffa6wm/mmdet3d.egg-info
  writing manifest file '/tmp/pip-pip-egg-info-lmffa6wm/mmdet3d.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-pip-egg-info-lmffa6wm/mmdet3d.egg-info/SOURCES.txt'
  Preparing metadata (setup.py) ... done
  Obtaining dependency information for lyft_dataset_sdk from https://files.pythonhosted.org/packages/52/d0/1203423cd24ae9da33a66736a75ee81f7f7e487dde5041b99a66f86a444d/lyft_dataset_sdk-0.0.8-py2.py3-none-any.whl.metadata
  Link requires a different Python (3.8.20 not in: '>=3.9'): https://files.pythonhosted.org/packages/49/38/8433925423d52727d286e1e7d6a234e8f2f084cf84a1942b122231c6ef76/networkx-3.2rc0-py3-none-any.whl (from https://pypi.org/simple/networkx/) (requires-python:>=3.9)
  Link requires a different Python (

In [17]:
!/opt/miniconda3/envs/bevfusion/bin/pip install cumm-cu113
!/opt/miniconda3/envs/bevfusion/bin/pip install spconv-cu113

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.6/22.6 MB 73.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 MB 68.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.6/22.6 MB 64.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: cumm-cu113
    Found existing installation: cumm-cu113 0.5.3
    Uninstalling cumm-cu113-0.5.3:
      Successfully uninstalled cumm-cu113-0.5.3


In [18]:
!/opt/miniconda3/envs/bevfusion/bin/mim download mmdet3d --config pointpillars_hv_secfpn_8xb6-160e_kitti-3d-car --dest .

processing pointpillars_hv_secfpn_8xb6-160e_kitti-3d-car...
downloading ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.4/18.4 MiB 9.4 MB/s eta 0:00:0000:0100:01
Successfully downloaded hv_pointpillars_secfpn_6x8_160e_kitti-3d-car_20220331_134606-d42d15ed.pth to /kaggle/working/mmdetection3d
Successfully dumped pointpillars_hv_secfpn_8xb6-160e_kitti-3d-car.py to /kaggle/working/mmdetection3d


**Verifying installation**

In [19]:
!/opt/miniconda3/envs/bevfusion/bin/python demo/pcd_demo.py demo/data/kitti/000008.bin pointpillars_hv_secfpn_8xb6-160e_kitti-3d-car.py hv_pointpillars_secfpn_6x8_160e_kitti-3d-car_20220331_134606-d42d15ed.pth

/kaggle/working/mmdetection3d/mmdet3d/models/dense_heads/anchor3d_head.py:94: UserWarning: dir_offset and dir_limit_offset will be depressed and be incorporated into box coder in the future
  warnings.warn(
Loads checkpoint by local backend from path: hv_pointpillars_secfpn_6x8_160e_kitti-3d-car_20220331_134606-d42d15ed.pth
09/17 17:18:21 - mmengine - WARNING - Failed to search registry with scope "mmdet3d" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet3d" is a correct scope, or whether the registry is initialized.
/opt/miniconda3/envs/bevfusion/lib/python3.8/site-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '
/opt/miniconda3/envs

# Setting up BEV-Fusion
CUDA toolkit and gcc were already installed and wired up earlier, so this
should now compile cleanly on the first try.

In [20]:
%cd projects/BEVFusion

/kaggle/working/mmdetection3d/projects/BEVFusion


In [ ]:
!ls
!cat README.md

In [21]:
%cd ../..

/kaggle/working/mmdetection3d


In [23]:
!/opt/miniconda3/bin/conda install -n bevfusion -y -c conda-forge libxcrypt

Channels:
 - conda-forge
 - defaults
 - nvidia/label/cuda-11.3.1
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/miniconda3/envs/bevfusion

  added / updated specs:
    - libxcrypt


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libxcrypt-4.4.38           |       h280c20c_0         100 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         100 KB

The following NEW packages will be INSTALLED:

  libxcrypt          conda-forge/linux-64::libxcrypt-4.4.38-h280c20c_0 



                                                                                
Preparing transaction: done
Verifying transaction: done
Executing transaction: done


In [26]:
!mkdir -p /lib64
!ln -sf /usr/lib/x86_64-linux-gnu/libm.so.6 /lib64/libm.so.6
!ln -sf /usr/lib/x86_64-linux-gnu/libmvec.so.1 /lib64/libmvec.so.1

In [36]:
!/opt/miniconda3/envs/bevfusion/bin/python projects/BEVFusion/setup.py develop

OSError: [Errno 2] No such file or directory